# NISAR L1 GSLC Product Tutorial

This tutorial demonstrates how to work with NISAR Level-1 Geocoded Single Look Complex (GSLC) products, including interferogram and coherence generation.

## Table of Contents
1. [Introduction to GSLC Products](#introduction)
2. [Understanding Granule Naming Convention](#granule-naming)
3. [Downloading Data from ASF DAAC](#downloading)
4. [Exploring the HDF5 Structure](#hdf5-structure)
5. [Working with GSLC Data](#gslc-data)
6. [Reading and Visualizing GSLC Data](#visualization)
7. [Interferogram Formation](#interferogram)
8. [Coherence Calculation](#coherence)
9. [Block Processing for Large Files](#block-processing)
10. [Complete Interferogram Generation Workflow](#complete-workflow)

## 1. Introduction to GSLC Products <a name="introduction"></a>

The NISAR L1 GSLC (Geocoded Single Look Complex) product contains focused SAR data that has been projected into a geographic coordinate system. Key characteristics:

- **Complex-valued data**: Contains both amplitude and phase information
- **Geocoded geometry**: Data is map-projected (UTM or Polar Stereographic)
- **Dual-frequency capable**: L-band (frequency A) and S-band (frequency B)
- **Multi-polarimetric**: Supports HH, HV, VH, VV polarizations
- **HDF5 format**: Hierarchical data structure
- **Large file sizes**: 20-40 GB per product (requires block processing)

GSLC products are essential for:
- Interferometric SAR (InSAR) processing in map coordinates
- Time series analysis
- Direct comparison with other geocoded data

In [ ]:
# Import required libraries
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy import ndimage
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Understanding Granule Naming Convention <a name="granule-naming"></a>

GSLC granule naming is similar to RSLC but indicates geocoded data:

```
NISAR_L1_PR_GSLC_009_172_A_008_2005_DHDH_A_20260109T024620_20260109T024654_X05009_N_F_J_001
```

| Component | Value | Description |
|-----------|-------|-------------|
| Mission | NISAR | NASA-ISRO SAR mission |
| Level | L1 | Processing level (Level-1) |
| Product Type | PR | Product type (PR = Product) |
| Product | GSLC | Geocoded Single Look Complex |
| Cycle Number | 009 | Mission cycle number |
| Relative Orbit | 172 | Relative orbit number |
| Orbit Direction | A | Ascending (A) or Descending (D) |
| Track Number | 008 | Track number |
| Frame Number | 2005 | Frame number within track |
| Frequency/Polarization | DHDH | Frequency and polarization |
| Production Type | A | Actual (A) vs Simulated |
| Start Time | 20260109T024620 | YYYYMMDDTHHMMSS (UTC) |
| End Time | 20260109T024654 | YYYYMMDDTHHMMSS (UTC) |

In [ ]:
# Example granule information
granule_id = "NISAR_L2_PR_GSLC_026_082_A_174_4005_DHDH_A_20260725T223100_20260725T223135_P05023_N_F_J_001"

def parse_granule_name(granule_id):
    """Parse NISAR GSLC granule name into components"""
    parts = granule_id.split('_')
    return {
        'mission': parts[0],
        'level': parts[1],
        'production_type': parts[2],
        'product_type': parts[3],
        'cycle': int(parts[4]),
        'track': int(parts[5]),
        'orbit_direction': parts[6],
        'frame': int(parts[7]),
        'range_bandwidth': parts[8],
        'polarization': parts[9],
        'single_or_mixed': parts[10],
        'start_time': parts[11],
        'end_time': parts[12],
        'crid': parts[13],
        'orbit_accuracy': parts[14],
        'full_or_partial_frame': parts[15],
        'processing_center': parts[16],
        'counter': parts[17]

    }

granule_info = parse_granule_name(granule_id)
print("Granule Information:")
for key, value in granule_info.items():
    print(f"  {key:20s}: {value}")

## 3. Downloading Data from ASF DAAC <a name="downloading"></a>

NISAR GSLC products are available from the Alaska Satellite Facility (ASF) DAAC.

### Option 1: Manual Download from ASF Vertex
1. Visit https://search.asf.alaska.edu/
2. Search for "NISAR" and filter by product type "GSLC"
3. Download the granule

For this tutorial, we'll assume you have two GSLC files for interferogram generation.

In [ ]:
# Specify paths to your GSLC files
# Update these paths to point to your downloaded GSLC files
gslc_file_1 = "/home/jovyan/data/NISAR/tutorial_data/NISAR_L2_PR_GSLC_025_082_A_174_4005_DHDH_A_20260713T223101_20260713T223136_P05023_N_F_J_001.h5"
gslc_file_2 = "/home/jovyan/data/NISAR/tutorial_data/NISAR_L2_PR_GSLC_026_082_A_174_4005_DHDH_A_20260725T223100_20260725T223135_P05023_N_F_J_001.h5"

# Check if files exist
for gslc_file in [gslc_file_1, gslc_file_2]:
    if os.path.exists(gslc_file):
        file_size = os.path.getsize(gslc_file) / (1024**3)  # Size in GB
        print(f"File found: {gslc_file}")
        print(f"  File size: {file_size:.2f} GB")
    else:
        print(f"File not found: {gslc_file}")
        print("Please update the path to your GSLC file.")

## 4. Exploring the HDF5 Structure <a name="hdf5-structure"></a>

NISAR GSLC products use HDF5 format with a hierarchical structure similar to RSLC but organized for geocoded data.

In [ ]:
def print_hdf5_structure(name, obj, indent=0):
    """Recursively print HDF5 structure"""
    spacing = '  ' * indent
    if isinstance(obj, h5py.Group):
        print(f"{spacing}{name}/ (Group)")
    elif isinstance(obj, h5py.Dataset):
        print(f"{spacing}{name} (Dataset): shape={obj.shape}, dtype={obj.dtype}")

# Open the file and explore top-level structure
with h5py.File(gslc_file_1, 'r') as f:
    print("\n=== Top-level HDF5 Structure ===")
    print_hdf5_structure('/', f)
    
    for key in f.keys():
        print_hdf5_structure(key, f[key], indent=1)
        if key == 'science':
            for key2 in f[key].keys():
                print_hdf5_structure(key2, f[key][key2], indent=2)
                if key2 == 'LSAR':
                    for key3 in f[key][key2].keys():
                        print_hdf5_structure(key3, f[key][key2][key3], indent=3)

## 5. Working with GSLC Data <a name="gslc-data"></a>

GSLC data is organized under `/science/LSAR/GSLC/grids/`. Key features:

- **Frequency A**: L-band (~24 cm wavelength, 1.26 GHz)
- **Frequency B**: S-band (~12 cm wavelength, 3.2 GHz) - if available
- **Polarizations**: HH, HV, VH, VV depending on acquisition mode
- **Map projection**: UTM or Polar Stereographic
- **Coordinates**: xCoordinates and yCoordinates in projected coordinate system

In [ ]:
with h5py.File(gslc_file_1, 'r') as f:
    grid_path = '/science/LSAR/GSLC/grids'
    
    print("\n=== Available Frequencies ===")
    for freq in f[grid_path].keys():
        print(f"\n{freq}:")
        freq_path = f"{grid_path}/{freq}"
        
        # List available polarizations
        if 'listOfPolarizations' in f[freq_path]:
            pols = f[freq_path]['listOfPolarizations'][()]
            if isinstance(pols, bytes):
                pols = pols.decode('utf-8')
            print(f"  Polarizations: {pols}")
        
        # Find datasets (polarization channels)
        print("  Available datasets:")
        for item in f[freq_path].keys():
            if isinstance(f[freq_path][item], h5py.Dataset):
                dataset = f[freq_path][item]
                print(f"    {item}: shape={dataset.shape}, dtype={dataset.dtype}")
        
        # Get coordinate information
        if 'xCoordinates' in f[freq_path]:
            x_coords = f[f'{freq_path}/xCoordinates'][:]
            y_coords = f[f'{freq_path}/yCoordinates'][:]
            print(f"  X range: {x_coords[0]:.2f} - {x_coords[-1]:.2f}")
            print(f"  Y range: {y_coords[0]:.2f} - {y_coords[-1]:.2f}")
            print(f"  Pixel spacing X: {np.mean(np.diff(x_coords)):.2f} m")
            print(f"  Pixel spacing Y: {np.mean(np.diff(y_coords)):.2f} m")

## 6. Reading and Visualizing GSLC Data <a name="visualization"></a>

GSLC files can be 20-40 GB in size, so we'll use subsetting and striding for visualization.

In [ ]:
def read_gslc_data(h5_file, frequency='frequencyA', polarization='HH', 
                   subset=None, stride=(1, 1)):
    """
    Read GSLC data with subsetting and striding options
    
    Parameters:
    -----------
    h5_file : str or h5py.File
        Path to GSLC file or open file handle
    frequency : str
        'frequencyA' or 'frequencyB'
    polarization : str
        Polarization channel (e.g., 'HH', 'HV', 'VV', 'VH')
    subset : tuple or None
        ((row_start, row_end), (col_start, col_end)) for spatial subset
    stride : tuple
        (row_stride, col_stride) for downsampling
    
    Returns:
    --------
    data : complex numpy array
        SLC data
    x_coords : array
        X coordinates of the data
    y_coords : array
        Y coordinates of the data
    """
    should_close = False
    if isinstance(h5_file, str):
        f = h5py.File(h5_file, 'r')
        should_close = True
    else:
        f = h5_file
    
    try:
        # Construct dataset path
        dataset_path = f'/science/LSAR/GSLC/grids/{frequency}/{polarization}'
        
        if dataset_path not in f:
            raise ValueError(f"Dataset not found: {dataset_path}")
        
        dataset = f[dataset_path]
        
        # Get full dimensions
        full_shape = dataset.shape
        
        # Determine slice indices
        if subset is None:
            row_slice = slice(None, None, stride[0])
            col_slice = slice(None, None, stride[1])
        else:
            row_slice = slice(subset[0][0], subset[0][1], stride[0])
            col_slice = slice(subset[1][0], subset[1][1], stride[1])
        
        # Read data
        data = dataset[row_slice, col_slice]
        
        # Read coordinates
        x_coords_full = f[f'/science/LSAR/GSLC/grids/{frequency}/xCoordinates'][:]
        y_coords_full = f[f'/science/LSAR/GSLC/grids/{frequency}/yCoordinates'][:]
        
        # Subset coordinates
        if subset is None:
            x_coords = x_coords_full[::stride[1]]
            y_coords = y_coords_full[::stride[0]]
        else:
            x_coords = x_coords_full[subset[1][0]:subset[1][1]:stride[1]]
            y_coords = y_coords_full[subset[0][0]:subset[0][1]:stride[0]]
        
        return data, x_coords, y_coords
    
    finally:
        if should_close:
            f.close()

# Example: Read a subset with stride for quick look
with h5py.File(gslc_file_1, 'r') as f:
    # Get full dimensions
    dataset_path = '/science/LSAR/GSLC/grids/frequencyA/HH'
    full_shape = f[dataset_path].shape
    print(f"Full image dimensions: {full_shape}")
    
    # Read center subset with stride for quick look
    row_center = full_shape[0] // 2
    col_center = full_shape[1] // 2
    window_size = 2000
    
    subset = (
        (max(0, row_center - window_size), min(full_shape[0], row_center + window_size)),
        (max(0, col_center - window_size), min(full_shape[1], col_center + window_size))
    )
    
    slc_data, x_coords, y_coords = read_gslc_data(f, frequency='frequencyA', polarization='HH',
                                                   subset=subset, stride=(2, 2))
    
    print(f"Loaded subset shape: {slc_data.shape}")
    print(f"Data type: {slc_data.dtype}")

In [ ]:
# Visualize the GSLC data
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]

# Amplitude
amplitude = np.abs(slc_data)
im1 = axes[0].imshow(amplitude, cmap='gray', aspect='auto', extent=extent)
axes[0].set_title('Amplitude')
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
plt.colorbar(im1, ax=axes[0])

# Amplitude in dB
amplitude_db = 20 * np.log10(amplitude + 1e-10)
im2 = axes[1].imshow(amplitude_db, cmap='gray', aspect='auto', extent=extent,
                     vmin=np.percentile(amplitude_db, 5),
                     vmax=np.percentile(amplitude_db, 95))
axes[1].set_title('Amplitude (dB)')
axes[1].set_xlabel('Easting (m)')
axes[1].set_ylabel('Northing (m)')
plt.colorbar(im2, ax=axes[1], label='dB')

# Phase
phase = np.angle(slc_data)
im3 = axes[2].imshow(phase, cmap='hsv', aspect='auto', extent=extent,
                     vmin=-np.pi, vmax=np.pi)
axes[2].set_title('Phase')
axes[2].set_xlabel('Easting (m)')
axes[2].set_ylabel('Northing (m)')
plt.colorbar(im3, ax=axes[2], label='radians')

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Data Statistics ===")
print(f"Mean amplitude: {np.mean(amplitude):.2f}")
print(f"Std amplitude: {np.std(amplitude):.2f}")
print(f"Min amplitude: {np.min(amplitude):.2f}")
print(f"Max amplitude: {np.max(amplitude):.2f}")

## 7. Interferogram Formation <a name="interferogram"></a>

An interferogram is formed by cross-multiplying two SLC images:

$$\text{Interferogram} = \text{SLC}_1 \times \text{SLC}_2^*$$

where $*$ denotes complex conjugate.

The interferometric phase is:
$$\phi = \arg(\text{Interferogram})$$

In [ ]:
def form_interferogram(slc1, slc2, multilook_window=(1, 1)):
    """
    Form interferogram from two SLC images with multi-looking
    
    Parameters:
    -----------
    slc1 : complex array
        Reference SLC image
    slc2 : complex array
        Secondary SLC image
    multilook_window : tuple
        (azimuth_looks, range_looks) for multi-looking
    
    Returns:
    --------
    ifg : complex array
        Complex interferogram (multi-looked)
    """
    # Form interferogram by cross multiplication
    ifg_full = slc1 * np.conj(slc2)
    
    # Multi-look if requested
    if multilook_window != (1, 1):
        az_looks, rg_looks = multilook_window
        
        # Calculate output dimensions
        out_rows = slc1.shape[0] // az_looks
        out_cols = slc1.shape[1] // rg_looks
        
        # Trim to multiple of window size
        trim_rows = out_rows * az_looks
        trim_cols = out_cols * rg_looks
        ifg_trim = ifg_full[:trim_rows, :trim_cols]
        
        # Reshape and average
        ifg = ifg_trim.reshape(out_rows, az_looks, out_cols, rg_looks).mean(axis=(1, 3))
    else:
        ifg = ifg_full
    
    return ifg

# Example: Form interferogram with subset data
with h5py.File(gslc_file_2, 'r') as f:
    slc_data2, _, _ = read_gslc_data(f, frequency='frequencyA', polarization='HH',
                                     subset=subset, stride=(2, 2))

# Form interferogram (no multi-looking for now)
ifg = form_interferogram(slc_data, slc_data2, multilook_window=(1, 1))

print(f"Interferogram shape: {ifg.shape}")
print(f"Interferogram dtype: {ifg.dtype}")

In [ ]:
# Visualize interferogram
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]

# Interferogram phase
ifg_phase = np.angle(ifg)
im1 = axes[0].imshow(ifg_phase, cmap='hsv', aspect='auto', extent=extent,
                     vmin=-np.pi, vmax=np.pi)
axes[0].set_title('Interferogram Phase')
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
plt.colorbar(im1, ax=axes[0], label='Phase (radians)')

# Interferogram amplitude
ifg_amp = np.abs(ifg)
im2 = axes[1].imshow(ifg_amp, cmap='gray', aspect='auto', extent=extent, vmin =0, vmax = 1)
axes[1].set_title('Interferogram Amplitude')
axes[1].set_xlabel('Easting (m)')
axes[1].set_ylabel('Northing (m)')
plt.colorbar(im2, ax=axes[1], label='Amplitude')

plt.tight_layout()
plt.show()

## 8. Coherence Calculation <a name="coherence"></a>

Interferometric coherence is a measure of correlation between two SLC images:

$$\gamma = \frac{|\sum_{i=1}^{N} \text{SLC}_1^i \times (\text{SLC}_2^i)^*|}{\sqrt{\sum_{i=1}^{N} |\text{SLC}_1^i|^2 \times \sum_{i=1}^{N} |\text{SLC}_2^i|^2}}$$

where $N$ is the number of pixels in the multi-looking window.

Coherence ranges from 0 (no correlation) to 1 (perfect correlation).

In [ ]:
def calculate_coherence(slc1, slc2, window_size=(5, 5)):
    """
    Calculate interferometric coherence
    
    Parameters:
    -----------
    slc1 : complex array
        Reference SLC image
    slc2 : complex array
        Secondary SLC image
    window_size : tuple
        (azimuth_window, range_window) for coherence estimation
    
    Returns:
    --------
    coherence : float array
        Coherence magnitude (0-1)
    """
    # Form interferogram
    ifg = slc1 * np.conj(slc2)
    
    # Calculate power
    power1 = np.abs(slc1) ** 2
    power2 = np.abs(slc2) ** 2
    
    # Apply uniform filter (moving average)
    # For complex interferogram, filter real and imaginary separately
    ifg_sum_real = ndimage.uniform_filter(np.real(ifg), size=window_size, mode='constant')
    ifg_sum_imag = ndimage.uniform_filter(np.imag(ifg), size=window_size, mode='constant')
    ifg_sum = ifg_sum_real + 1j * ifg_sum_imag
    
    power1_sum = ndimage.uniform_filter(power1, size=window_size, mode='constant')
    power2_sum = ndimage.uniform_filter(power2, size=window_size, mode='constant')
    
    # Calculate coherence
    coherence = np.abs(ifg_sum) / (np.sqrt(power1_sum * power2_sum) + 1e-10)
    
    # Clip to [0, 1] to handle numerical issues
    coherence = np.clip(coherence, 0, 1)
    
    return coherence

# Calculate coherence
coherence = calculate_coherence(slc_data, slc_data2, window_size=(5, 5))

print(f"Coherence shape: {coherence.shape}")
print(f"Coherence range: {coherence.min():.3f} - {coherence.max():.3f}")
print(f"Mean coherence: {coherence.mean():.3f}")

In [ ]:
# Visualize coherence
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]

# Coherence map
im1 = axes[0].imshow(coherence, cmap='gray', aspect='auto', extent=extent,
                     vmin=0, vmax=1)
axes[0].set_title('Interferometric Coherence')
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
plt.colorbar(im1, ax=axes[0], label='Coherence')

# Coherence histogram
axes[1].hist(coherence.ravel(), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Coherence')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Coherence Distribution')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(coherence.mean(), color='gray', linestyle='--', linewidth=2,
                label=f'Mean: {coherence.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Block Processing for Large Files <a name="block-processing"></a>

For large GSLC files (20-40 GB), we need to process data in blocks to avoid memory issues. 

**Key considerations:**
1. **Block overlap**: Ensure blocks overlap by at least half the multi-looking window size to avoid edge artifacts
2. **Memory management**: Process one block at a time
3. **Output writing**: Write results incrementally to HDF5 or memory-mapped arrays

In [ ]:
def generate_blocks(full_shape, block_size=(2000, 2000), overlap=(50, 50)):
    """
    Generate block indices with overlap for processing large arrays
    
    Parameters:
    -----------
    full_shape : tuple
        (rows, cols) of full array
    block_size : tuple
        (block_rows, block_cols) size of each block
    overlap : tuple
        (overlap_rows, overlap_cols) overlap between blocks
    
    Yields:
    -------
    block_info : dict
        Dictionary with block indices:
        - 'read': (row_start, row_end, col_start, col_end) for reading with overlap
        - 'write': (row_start, row_end, col_start, col_end) for writing (trim overlap)
        - 'trim': (trim_top, trim_bottom, trim_left, trim_right) for trimming overlap
        - 'block_id': (block_row, block_col)
    """
    rows, cols = full_shape
    block_rows, block_cols = block_size
    overlap_rows, overlap_cols = overlap
    
    # Calculate number of blocks
    n_row_blocks = int(np.ceil(rows / block_rows))
    n_col_blocks = int(np.ceil(cols / block_cols))
    
    for i_block in range(n_row_blocks):
        for j_block in range(n_col_blocks):
            # Calculate write indices (no overlap)
            write_row_start = i_block * block_rows
            write_row_end = min((i_block + 1) * block_rows, rows)
            write_col_start = j_block * block_cols
            write_col_end = min((j_block + 1) * block_cols, cols)
            
            # Calculate read indices (with overlap)
            read_row_start = max(0, write_row_start - overlap_rows)
            read_row_end = min(rows, write_row_end + overlap_rows)
            read_col_start = max(0, write_col_start - overlap_cols)
            read_col_end = min(cols, write_col_end + overlap_cols)
            
            # Calculate trim indices (how much to trim from read block)
            trim_top = write_row_start - read_row_start
            trim_bottom = read_row_end - write_row_end
            trim_left = write_col_start - read_col_start
            trim_right = read_col_end - write_col_end
            
            yield {
                'read': (read_row_start, read_row_end, read_col_start, read_col_end),
                'write': (write_row_start, write_row_end, write_col_start, write_col_end),
                'trim': (trim_top, trim_bottom, trim_left, trim_right),
                'block_id': (i_block, j_block)
            }

# Example: Generate blocks for a sample array
sample_shape = (10000, 8000)
block_size = (2000, 2000)
overlap = (100, 100)  # Overlap should be >= half of multi-looking window

blocks = list(generate_blocks(sample_shape, block_size=block_size, overlap=overlap))
print(f"Total blocks to process: {len(blocks)}")
print(f"\nFirst block info:")
print(f"  Read indices: {blocks[0]['read']}")
print(f"  Write indices: {blocks[0]['write']}")
print(f"  Trim: {blocks[0]['trim']}")
print(f"  Block ID: {blocks[0]['block_id']}")

In [ ]:
def process_gslc_block(slc1_block, slc2_block, multilook_window=(3, 3), 
                       coherence_window=(5, 5), trim=(0, 0, 0, 0)):
    """
    Process a single block: form interferogram and calculate coherence
    
    Parameters:
    -----------
    slc1_block : complex array
        Reference SLC block (with overlap)
    slc2_block : complex array
        Secondary SLC block (with overlap)
    multilook_window : tuple
        (azimuth_looks, range_looks) for multi-looking
    coherence_window : tuple
        (azimuth_window, range_window) for coherence estimation
    trim : tuple
        (trim_top, trim_bottom, trim_left, trim_right) to remove overlap
    
    Returns:
    --------
    ifg_ml : complex array
        Multi-looked interferogram (trimmed)
    coherence : float array
        Coherence magnitude (trimmed)
    """
    # Form interferogram with multi-looking
    ifg_ml = form_interferogram(slc1_block, slc2_block, multilook_window=multilook_window)
    
    # Calculate coherence (on multi-looked data)
    slc1_ml = form_interferogram(slc1_block, slc1_block, multilook_window=multilook_window)
    slc2_ml = form_interferogram(slc2_block, slc2_block, multilook_window=multilook_window)
    
    # Note: For coherence, we need the SLC data, not interferogram
    # So we recalculate on multi-looked SLCs
    az_looks, rg_looks = multilook_window
    out_rows = slc1_block.shape[0] // az_looks
    out_cols = slc1_block.shape[1] // rg_looks
    trim_rows = out_rows * az_looks
    trim_cols = out_cols * rg_looks
    
    slc1_trim = slc1_block[:trim_rows, :trim_cols]
    slc2_trim = slc2_block[:trim_rows, :trim_cols]
    
    slc1_ml_proper = slc1_trim.reshape(out_rows, az_looks, out_cols, rg_looks).mean(axis=(1, 3))
    slc2_ml_proper = slc2_trim.reshape(out_rows, az_looks, out_cols, rg_looks).mean(axis=(1, 3))
    
    coherence = calculate_coherence(slc1_ml_proper, slc2_ml_proper, window_size=coherence_window)
    
    # Trim overlap from multi-looked results
    trim_top, trim_bottom, trim_left, trim_right = trim
    
    # Adjust trim for multi-looking
    trim_top_ml = trim_top // az_looks
    trim_left_ml = trim_left // rg_looks
    
    if trim_bottom == 0:
        row_end = None
    else:
        row_end = -(trim_bottom // az_looks)
    
    if trim_right == 0:
        col_end = None
    else:
        col_end = -(trim_right // rg_looks)
    
    ifg_ml_trimmed = ifg_ml[trim_top_ml:row_end, trim_left_ml:col_end]
    coherence_trimmed = coherence[trim_top_ml:row_end, trim_left_ml:col_end]
    
    return ifg_ml_trimmed, coherence_trimmed

# Test with sample data
sample_ifg, sample_coh = process_gslc_block(
    slc_data, slc_data2, 
    multilook_window=(3, 3), 
    coherence_window=(5, 5),
    trim=(0, 0, 0, 0)
)

print(f"Multi-looked interferogram shape: {sample_ifg.shape}")
print(f"Coherence shape: {sample_coh.shape}")

## 10. Complete Interferogram Generation Workflow <a name="complete-workflow"></a>

This function processes two full GSLC products in blocks to generate interferogram and coherence.

In [ ]:
def generate_interferogram_coherence(
    gslc_file_1, 
    gslc_file_2, 
    output_ifg_file,
    output_coh_file,
    frequency='frequencyA', 
    polarization='HH',
    multilook_window=(3, 3),
    coherence_window=(5, 5),
    block_size=(2000, 2000),
    overlap=None
):
    """
    Generate interferogram and coherence from two GSLC products using block processing
    
    Parameters:
    -----------
    gslc_file_1 : str
        Path to reference GSLC file
    gslc_file_2 : str
        Path to secondary GSLC file
    output_ifg_file : str
        Path to output interferogram file (HDF5)
    output_coh_file : str
        Path to output coherence file (HDF5)
    frequency : str
        Frequency to process ('frequencyA' or 'frequencyB')
    polarization : str
        Polarization to process ('HH', 'HV', 'VH', 'VV')
    multilook_window : tuple
        (azimuth_looks, range_looks) for multi-looking
    coherence_window : tuple
        (azimuth_window, range_window) for coherence estimation
    block_size : tuple
        (block_rows, block_cols) size of processing blocks
    overlap : tuple or None
        (overlap_rows, overlap_cols) overlap between blocks
        If None, automatically set to max(multilook_window, coherence_window)
    
    Returns:
    --------
    None (writes results to files)
    """
    # Determine overlap if not specified
    if overlap is None:
        overlap = (
            max(multilook_window[0], coherence_window[0]),
            max(multilook_window[1], coherence_window[1])
        )
    
    print(f"Processing interferogram and coherence")
    print(f"  Frequency: {frequency}")
    print(f"  Polarization: {polarization}")
    print(f"  Multi-look window: {multilook_window}")
    print(f"  Coherence window: {coherence_window}")
    print(f"  Block size: {block_size}")
    print(f"  Overlap: {overlap}")
    
    # Open both GSLC files
    with h5py.File(gslc_file_1, 'r') as f1, h5py.File(gslc_file_2, 'r') as f2:
        # Get dataset paths
        dataset_path = f'/science/LSAR/GSLC/grids/{frequency}/{polarization}'
        
        # Check if datasets exist
        if dataset_path not in f1 or dataset_path not in f2:
            raise ValueError(f"Dataset not found in one or both files: {dataset_path}")
        
        # Get full dimensions
        full_shape = f1[dataset_path].shape
        print(f"\nFull image dimensions: {full_shape}")
        
        # Calculate output dimensions after multi-looking
        az_looks, rg_looks = multilook_window
        out_shape = (full_shape[0] // az_looks, full_shape[1] // rg_looks)
        print(f"Output dimensions (multi-looked): {out_shape}")
        
        # Get coordinates
        x_coords_full = f1[f'/science/LSAR/GSLC/grids/{frequency}/xCoordinates'][:]
        y_coords_full = f1[f'/science/LSAR/GSLC/grids/{frequency}/yCoordinates'][:]
        
        # Multi-look coordinates (take every Nth coordinate)
        x_coords_ml = x_coords_full[::rg_looks]
        y_coords_ml = y_coords_full[::az_looks]
        
        # Create output HDF5 files
        with h5py.File(output_ifg_file, 'w') as fout_ifg, \
             h5py.File(output_coh_file, 'w') as fout_coh:
            
            # Create datasets for output
            ifg_dataset = fout_ifg.create_dataset(
                'interferogram',
                shape=out_shape,
                dtype=np.complex64,
                chunks=(min(1000, out_shape[0]), min(1000, out_shape[1])),
                compression='gzip'
            )
            
            coh_dataset = fout_coh.create_dataset(
                'coherence',
                shape=out_shape,
                dtype=np.float32,
                chunks=(min(1000, out_shape[0]), min(1000, out_shape[1])),
                compression='gzip'
            )
            
            # Write coordinates
            fout_ifg.create_dataset('xCoordinates', data=x_coords_ml)
            fout_ifg.create_dataset('yCoordinates', data=y_coords_ml)
            fout_coh.create_dataset('xCoordinates', data=x_coords_ml)
            fout_coh.create_dataset('yCoordinates', data=y_coords_ml)
            
            # Write metadata
            ifg_dataset.attrs['frequency'] = frequency
            ifg_dataset.attrs['polarization'] = polarization
            ifg_dataset.attrs['multilook_azimuth'] = multilook_window[0]
            ifg_dataset.attrs['multilook_range'] = multilook_window[1]
            
            coh_dataset.attrs['frequency'] = frequency
            coh_dataset.attrs['polarization'] = polarization
            coh_dataset.attrs['multilook_azimuth'] = multilook_window[0]
            coh_dataset.attrs['multilook_range'] = multilook_window[1]
            coh_dataset.attrs['coherence_window_azimuth'] = coherence_window[0]
            coh_dataset.attrs['coherence_window_range'] = coherence_window[1]
            
            # Generate blocks
            blocks = list(generate_blocks(full_shape, block_size=block_size, overlap=overlap))
            total_blocks = len(blocks)
            print(f"\nProcessing {total_blocks} blocks...")
            
            # Process each block
            for idx, block_info in enumerate(blocks):
                # Extract block info
                read_indices = block_info['read']
                write_indices = block_info['write']
                trim = block_info['trim']
                block_id = block_info['block_id']
                
                # Read blocks from both SLC files
                r_start, r_end, c_start, c_end = read_indices
                slc1_block = f1[dataset_path][r_start:r_end, c_start:c_end]
                slc2_block = f2[dataset_path][r_start:r_end, c_start:c_end]
                
                # Process block
                ifg_block, coh_block = process_gslc_block(
                    slc1_block, slc2_block,
                    multilook_window=multilook_window,
                    coherence_window=coherence_window,
                    trim=trim
                )
                
                # Write to output (adjust indices for multi-looking)
                w_r_start, w_r_end, w_c_start, w_c_end = write_indices
                out_r_start = w_r_start // az_looks
                out_r_end = w_r_end // az_looks
                out_c_start = w_c_start // rg_looks
                out_c_end = w_c_end // rg_looks
                
                ifg_dataset[out_r_start:out_r_end, out_c_start:out_c_end] = ifg_block
                coh_dataset[out_r_start:out_r_end, out_c_start:out_c_end] = coh_block
                
                # Print progress
                if (idx + 1) % 10 == 0 or (idx + 1) == total_blocks:
                    progress = 100 * (idx + 1) / total_blocks
                    print(f"  Progress: {idx+1}/{total_blocks} blocks ({progress:.1f}%)")
    
    print(f"\nProcessing complete!")
    print(f"  Interferogram saved to: {output_ifg_file}")
    print(f"  Coherence saved to: {output_coh_file}")

In [ ]:
# Example usage: Generate interferogram and coherence
# Uncomment and adjust parameters as needed

# generate_interferogram_coherence(
#     gslc_file_1='path/to/reference/GSLC_file.h5',
#     gslc_file_2='path/to/secondary/GSLC_file.h5',
#     output_ifg_file='interferogram.h5',
#     output_coh_file='coherence.h5',
#     frequency='frequencyA',
#     polarization='HH',
#     multilook_window=(3, 3),
#     coherence_window=(5, 5),
#     block_size=(2000, 2000)
# )

### Visualize Results

In [ ]:
def visualize_results(ifg_file, coh_file, stride=(10, 10)):
    """
    Visualize interferogram and coherence results
    
    Parameters:
    -----------
    ifg_file : str
        Path to interferogram HDF5 file
    coh_file : str
        Path to coherence HDF5 file
    stride : tuple
        (row_stride, col_stride) for downsampling visualization
    """
    with h5py.File(ifg_file, 'r') as f_ifg, h5py.File(coh_file, 'r') as f_coh:
        # Read data with stride
        ifg = f_ifg['interferogram'][::stride[0], ::stride[1]]
        coh = f_coh['coherence'][::stride[0], ::stride[1]]
        
        # Read coordinates
        x_coords = f_ifg['xCoordinates'][::stride[1]]
        y_coords = f_ifg['yCoordinates'][::stride[0]]
        
        extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]
        
        # Create figure
        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        
        # Interferogram phase
        ifg_phase = np.angle(ifg)
        im1 = axes[0, 0].imshow(ifg_phase, cmap='hsv', aspect='auto', extent=extent,
                                vmin=-np.pi, vmax=np.pi)
        axes[0, 0].set_title('Interferogram Phase')
        axes[0, 0].set_xlabel('Easting (m)')
        axes[0, 0].set_ylabel('Northing (m)')
        plt.colorbar(im1, ax=axes[0, 0], label='Phase (radians)')
        
        # Interferogram amplitude
        ifg_amp = np.abs(ifg)
        im2 = axes[0, 1].imshow(ifg_amp, cmap='gray', aspect='auto', extent=extent)
        axes[0, 1].set_title('Interferogram Amplitude')
        axes[0, 1].set_xlabel('Easting (m)')
        axes[0, 1].set_ylabel('Northing (m)')
        plt.colorbar(im2, ax=axes[0, 1], label='Amplitude')
        
        # Coherence
        im3 = axes[1, 0].imshow(coh, cmap='jet', aspect='auto', extent=extent,
                                vmin=0, vmax=1)
        axes[1, 0].set_title('Coherence')
        axes[1, 0].set_xlabel('Easting (m)')
        axes[1, 0].set_ylabel('Northing (m)')
        plt.colorbar(im3, ax=axes[1, 0], label='Coherence')
        
        # Coherence histogram
        axes[1, 1].hist(coh.ravel(), bins=50, edgecolor='black', alpha=0.7)
        axes[1, 1].set_xlabel('Coherence')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Coherence Distribution')
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].axvline(coh.mean(), color='red', linestyle='--', linewidth=2,
                          label=f'Mean: {coh.mean():.3f}')
        axes[1, 1].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print("\n=== Statistics ===")
        print(f"Interferogram shape: {ifg.shape}")
        print(f"Coherence range: {coh.min():.3f} - {coh.max():.3f}")
        print(f"Mean coherence: {coh.mean():.3f}")

# Example usage:
# visualize_results('interferogram.h5', 'coherence.h5', stride=(10, 10))

## Summary

This tutorial covered:

1. **GSLC Product Overview**: Understanding geocoded SLC data structure
2. **Granule Naming**: Decoding product identifiers
3. **Data Access**: Opening and navigating HDF5 files
4. **Visualization**: Reading and displaying complex SAR data
5. **Interferogram Formation**: Cross-multiplication with multi-looking
6. **Coherence Calculation**: Estimating interferometric coherence
7. **Block Processing**: Handling large files (20-40 GB) with overlapping blocks
8. **Complete Workflow**: End-to-end interferogram and coherence generation

### Key Points:

- **Block overlap** is critical to avoid edge artifacts in multi-looking
- Overlap should be at least as large as the largest processing window
- Memory-efficient processing allows handling of very large GSLC products
- Results are written incrementally to HDF5 files

For more information:
- NISAR Mission: https://nisar.jpl.nasa.gov/
- ASF DAAC: https://search.asf.alaska.edu/